# LeetCode 1551 — Minimum Operations to Make Array Equal bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-make-array-equal/ · Độ khó: **Medium** · Chủ đề: Math

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu bài toán "chuyển đơn vị giữa các phần tử" và nhận ra **tổng của mảng không đổi** sau mỗi thao tác ⇒ giá trị đích bắt buộc là **trung bình cộng**.
2. Viết baseline **mô phỏng** (mỗi bước chuyển 1 đơn vị từ phần tử lớn nhất sang nhỏ nhất) để kiểm chứng.
3. Chuyển từ mô phỏng sang công thức "**tổng phần dư phía trên trung bình**" và tính nó trong `O(n)` chỉ bằng nửa đầu mảng.
4. Rút gọn thành công thức đóng `n * n // 4` và chứng minh được nó đúng với cả `n` chẵn lẫn lẻ.
5. Tránh các bẫy: chia số thực `/`, quên chia đôi tổng khoảng cách, chọn sai giá trị đích khi `n` chẵn.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Có một mảng `arr` dài `n` với `arr[i] = 2*i + 1` (tức là `[1, 3, 5, ..., 2n−1]` — các số lẻ đầu tiên).

Mỗi **thao tác**: chọn hai chỉ số `x`, `y` rồi làm `arr[x] -= 1` và `arr[y] += 1` (chuyển 1 đơn vị từ `x` sang `y`). Đề đảm bảo luôn có cách làm cho mọi phần tử bằng nhau. Trả về **số thao tác ít nhất** để mọi phần tử của `arr` bằng nhau.

**Ví dụ 1:** `n = 3` → `2`. `arr = [1, 3, 5]`: lần 1 chọn `x = 2, y = 0` → `[2, 3, 4]`; lần 2 lặp lại → `[3, 3, 3]`.

**Ví dụ 2:** `n = 6` → `9`.

**Ràng buộc:** `1 ≤ n ≤ 10⁴`.

In [1]:
import heapq
import random
import time

examples = [(3, 2), (6, 9)]
for n, want in examples:
    arr = [2 * i + 1 for i in range(n)]
    print(f"n = {n}: arr = {arr} -> mong đợi {want}")

n = 3: arr = [1, 3, 5] -> mong đợi 2
n = 6: arr = [1, 3, 5, 7, 9, 11] -> mong đợi 9


Thêm vài giá trị nhỏ để có cảm giác (sẽ được kiểm chứng bằng mô phỏng ở mục 5 và mục 12): `n = 1 → 0`, `n = 2 → 1`, `n = 4 → 4`, `n = 5 → 6`.

In [2]:
small_known = {1: 0, 2: 1, 3: 2, 4: 4, 5: 6, 6: 9}
for n, v in small_known.items():
    print(n, [2 * i + 1 for i in range(n)], "->", v)

1 [1] -> 0
2 [1, 3] -> 1
3 [1, 3, 5] -> 2
4 [1, 3, 5, 7] -> 4
5 [1, 3, 5, 7, 9] -> 6
6 [1, 3, 5, 7, 9, 11] -> 9


## 3. Kiến thức nền

### 3.1 List comprehension để dựng mảng

`[2 * i + 1 for i in range(n)]` sinh nhanh mảng các số lẻ. `range(n)` chạy `i = 0, 1, ..., n−1`.

In [3]:
n = 7
arr = [2 * i + 1 for i in range(n)]
print(arr, "| tổng =", sum(arr), "| n² =", n * n)

[1, 3, 5, 7, 9, 11, 13] | tổng = 49 | n² = 49


### 3.2 Tổng `n` số lẻ đầu tiên bằng `n²`

`1 + 3 + 5 + ... + (2n−1) = n²`. Do đó trung bình cộng của mảng là `n² / n = n` — một **số nguyên**. Đây là lý do đề nói "luôn làm được".

In [4]:
for n in range(1, 9):
    arr = [2 * i + 1 for i in range(n)]
    print(f"n={n}: sum={sum(arr):>3}, n²={n*n:>3}, trung bình={sum(arr)//n}")

n=1: sum=  1, n²=  1, trung bình=1
n=2: sum=  4, n²=  4, trung bình=2
n=3: sum=  9, n²=  9, trung bình=3
n=4: sum= 16, n²= 16, trung bình=4
n=5: sum= 25, n²= 25, trung bình=5
n=6: sum= 36, n²= 36, trung bình=6
n=7: sum= 49, n²= 49, trung bình=7
n=8: sum= 64, n²= 64, trung bình=8


### 3.3 Chia nguyên `//` và chia thực `/`

`/` luôn trả về `float` (ví dụ `9 / 4 = 2.25`), còn `//` là chia lấy phần nguyên (làm tròn xuống). LeetCode yêu cầu trả `int` nên phải dùng `//`.

In [5]:
print(9 / 4, type(9 / 4))
print(9 // 4, type(9 // 4))
print(36 / 4, 36 // 4)

2.25 <class 'float'>
2 <class 'int'>
9.0 9


### 3.4 Hàm `sum` với generator và `abs`

`sum(abs(x - t) for x in arr)` tính tổng khoảng cách từ các phần tử tới `t` mà không cần tạo list trung gian.

In [6]:
arr = [1, 3, 5, 7]
t = 4
print("khoảng cách từng phần tử:", [abs(x - t) for x in arr])
print("tổng khoảng cách:", sum(abs(x - t) for x in arr))

khoảng cách từng phần tử: [3, 1, 1, 3]
tổng khoảng cách: 8


### 3.5 `heapq` — lấy phần tử nhỏ nhất / lớn nhất nhanh

`heapq` là min-heap. Để có max-heap ta lưu số âm. Dùng cho bản mô phỏng: mỗi bước lấy phần tử lớn nhất và nhỏ nhất.

In [7]:
h = [5, 1, 3]
heapq.heapify(h)
print("nhỏ nhất:", heapq.heappop(h))
mx = [-x for x in [5, 1, 3]]
heapq.heapify(mx)
print("lớn nhất:", -heapq.heappop(mx))

nhỏ nhất: 1
lớn nhất: 5


## 4. Ý tưởng nền tảng / trực giác

**Quan sát 1 — tổng bất biến.** Mỗi thao tác trừ 1 ở chỗ này và cộng 1 ở chỗ khác ⇒ `sum(arr)` không bao giờ thay đổi. Khi mọi phần tử bằng nhau và bằng `t` thì `n * t = sum(arr) = n²` ⇒ **`t = n`**. Không có lựa chọn nào khác cho giá trị đích!

**Quan sát 2 — đếm "phần dư".** Mỗi phần tử lớn hơn `n` phải **giảm** đúng `arr[i] − n` đơn vị; mỗi phần tử nhỏ hơn `n` phải **tăng** đúng `n − arr[i]` đơn vị. Mỗi thao tác giảm 1 đơn vị ở một chỗ và tăng 1 đơn vị ở chỗ khác.

**Tính tay với `n = 4`:** `arr = [1, 3, 5, 7]`, đích `t = 4`.

| i | arr[i] | arr[i] − 4 | Vai trò |
|---|--------|-----------|---------|
| 0 | 1 | −3 | cần nhận 3 |
| 1 | 3 | −1 | cần nhận 1 |
| 2 | 5 | +1 | cần cho 1 |
| 3 | 7 | +3 | cần cho 3 |

Tổng phần dư phía trên = `1 + 3 = 4` = tổng phần thiếu phía dưới. Mỗi thao tác chỉ chuyển được 1 đơn vị ⇒ cần ít nhất 4 thao tác, và 4 là đủ (chuyển 3 lần từ 7 sang 1, 1 lần từ 5 sang 3). Đáp án `4`.

**Tính tay với `n = 5`:** `arr = [1, 3, 5, 7, 9]`, đích `5`. Phần thiếu: `4 + 2 = 6` ⇒ đáp án `6`. Phần tử giữa (`5`) đã đúng, không cần đụng.

In [8]:
def excess_table(n):
    arr = [2 * i + 1 for i in range(n)]
    for i, x in enumerate(arr):
        d = x - n
        role = "cần nhận" if d < 0 else ("cần cho" if d > 0 else "đã đúng")
        print(f"i={i}: arr[i]={x:>2}, lệch={d:+d} -> {role} {abs(d)}")
    print("tổng phần dư phía trên =", sum(x - n for x in arr if x > n))

excess_table(4)
print()
excess_table(5)

i=0: arr[i]= 1, lệch=-3 -> cần nhận 3
i=1: arr[i]= 3, lệch=-1 -> cần nhận 1
i=2: arr[i]= 5, lệch=+1 -> cần cho 1
i=3: arr[i]= 7, lệch=+3 -> cần cho 3
tổng phần dư phía trên = 4

i=0: arr[i]= 1, lệch=-4 -> cần nhận 4
i=1: arr[i]= 3, lệch=-2 -> cần nhận 2
i=2: arr[i]= 5, lệch=+0 -> đã đúng 0
i=3: arr[i]= 7, lệch=+2 -> cần cho 2
i=4: arr[i]= 9, lệch=+4 -> cần cho 4
tổng phần dư phía trên = 6


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng thao tác

**Ý tưởng.** Làm đúng như đề: lặp lại "lấy phần tử lớn nhất, trừ 1; lấy phần tử nhỏ nhất, cộng 1" cho tới khi lớn nhất = nhỏ nhất. Đếm số lần.

Chiến lược "lớn nhất → nhỏ nhất" không bao giờ làm phí thao tác: phần tử lớn nhất chắc chắn > `n` và nhỏ nhất chắc chắn < `n` (khi chưa bằng nhau), nên mỗi bước giảm tổng phần dư đúng 1.

Bản đơn giản nhất dùng `max`/`min` trên list (mỗi bước `O(n)`).

In [9]:
def simulate_naive(n):
    arr = [2 * i + 1 for i in range(n)]
    ops = 0
    while max(arr) != min(arr):
        arr[arr.index(max(arr))] -= 1
        arr[arr.index(min(arr))] += 1
        ops += 1
    return ops

for n, want in examples:
    print(n, simulate_naive(n), "mong đợi", want)
print({n: simulate_naive(n) for n in range(1, 11)})

3 2 mong đợi 2
6 9 mong đợi 9
{1: 0, 2: 1, 3: 2, 4: 4, 5: 6, 6: 9, 7: 12, 8: 16, 9: 20, 10: 25}


Bản mô phỏng bằng heap: dùng một max-heap và một min-heap là rườm rà, nên ta dùng mẹo **ghép cặp đối xứng**: phần tử `i` và `n−1−i` có tổng `2n`, nên chỉ cần chuyển giữa đúng hai phần tử này. Đây vẫn là mô phỏng (từng đơn vị một), nhưng dễ kiểm tra hơn.

In [10]:
def simulate_pairs(n):
    arr = [2 * i + 1 for i in range(n)]
    ops = 0
    i, j = 0, n - 1
    while i < j:
        while arr[i] < arr[j]:          # chuyển 1 đơn vị từ j sang i
            arr[j] -= 1
            arr[i] += 1
            ops += 1
        i += 1
        j -= 1
    assert len(set(arr)) == 1, arr
    return ops

print({n: simulate_pairs(n) for n in range(1, 11)})

{1: 0, 2: 1, 3: 2, 4: 4, 5: 6, 6: 9, 7: 12, 8: 16, 9: 20, 10: 25}


**Độ phức tạp.** Số thao tác là khoảng `n²/4`. Bản `simulate_naive` tốn `O(n)` cho mỗi thao tác ⇒ `O(n³)`; bản `simulate_pairs` tốn `O(1)` mỗi thao tác ⇒ `O(n²)`. Với `n = 10⁴`, `n²/4 = 25 triệu` vòng lặp Python — chạy được nhưng chậm (vài giây), còn bản `O(n³)` thì không thể.

**Nút thắt:** ta đang **đếm từng đơn vị một**. Trong khi số đơn vị cần chuyển giữa cặp `(i, n−1−i)` có thể tính ngay bằng một phép trừ.

In [11]:
for n in [50, 200]:
    t0 = time.perf_counter()
    a = simulate_naive(n)
    t1 = time.perf_counter()
    b = simulate_pairs(n)
    t2 = time.perf_counter()
    print(f"n={n}: naive={a} ({t1-t0:.3f}s), pairs={b} ({t2-t1:.4f}s)")

n=50: naive=625 (0.002s), pairs=625 (0.0001s)
n=200: naive=10000 (0.100s), pairs=10000 (0.0008s)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (giá trị đích cố định).** Tổng không đổi ⇒ đích là `n`.

**Insight 2 (cận dưới = cận trên).** Gọi `E = Σ max(0, arr[i] − n)` là *tổng phần dư*. Mỗi thao tác giảm `E` **tối đa 1** (chỉ một phần tử bị trừ 1). Khi xong, `E = 0`. Vậy cần **ít nhất `E`** thao tác. Ngược lại, luôn có cách giảm `E` đúng 1 mỗi bước (chuyển từ một phần tử > n sang một phần tử < n) ⇒ **`E` thao tác là đủ**. Đáp án = `E`.

**Insight 3 (chỉ cần nửa đầu).** Mảng đối xứng quanh `n`: `arr[i] + arr[n−1−i] = 2n`. Phần dư của nửa sau đúng bằng phần thiếu của nửa đầu, nên

`E = Σ_{i=0}^{n//2 − 1} (n − (2i + 1))`.

**Insight 4 (công thức đóng).**
- `n` lẻ, `n = 2k+1`: phần thiếu là `2k, 2k−2, ..., 2` ⇒ tổng `k(k+1) = (n²−1)/4`.
- `n` chẵn, `n = 2k`: phần thiếu là `2k−1, 2k−3, ..., 1` ⇒ tổng `k² = n²/4`.

Cả hai trường hợp đều bằng **`n * n // 4`** (vì `(n²−1)/4 = ⌊n²/4⌋` khi `n` lẻ).

**Quy tắc nhớ:** *Nếu* bài cho phép "chuyển 1 đơn vị giữa hai phần tử" *thì* đáp án = tổng phần dư so với trung bình = `Σ|arr[i] − avg| / 2`.

In [12]:
for n in range(1, 13):
    arr = [2 * i + 1 for i in range(n)]
    E = sum(max(0, x - n) for x in arr)
    half = sum(n - (2 * i + 1) for i in range(n // 2))
    print(f"n={n:>2}: E={E:>3}, nửa đầu={half:>3}, n*n//4={n*n//4:>3}, Σ|.|/2={sum(abs(x-n) for x in arr)//2:>3}")

n= 1: E=  0, nửa đầu=  0, n*n//4=  0, Σ|.|/2=  0
n= 2: E=  1, nửa đầu=  1, n*n//4=  1, Σ|.|/2=  1
n= 3: E=  2, nửa đầu=  2, n*n//4=  2, Σ|.|/2=  2
n= 4: E=  4, nửa đầu=  4, n*n//4=  4, Σ|.|/2=  4
n= 5: E=  6, nửa đầu=  6, n*n//4=  6, Σ|.|/2=  6
n= 6: E=  9, nửa đầu=  9, n*n//4=  9, Σ|.|/2=  9
n= 7: E= 12, nửa đầu= 12, n*n//4= 12, Σ|.|/2= 12
n= 8: E= 16, nửa đầu= 16, n*n//4= 16, Σ|.|/2= 16
n= 9: E= 20, nửa đầu= 20, n*n//4= 20, Σ|.|/2= 20
n=10: E= 25, nửa đầu= 25, n*n//4= 25, Σ|.|/2= 25
n=11: E= 30, nửa đầu= 30, n*n//4= 30, Σ|.|/2= 30
n=12: E= 36, nửa đầu= 36, n*n//4= 36, Σ|.|/2= 36


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tổng khoảng cách tới trung bình (toàn mảng), `O(n)`

Dựng mảng, tính trung bình `t = sum // n`, trả `Σ|arr[i] − t| / 2`. Chia 2 vì mỗi thao tác "sửa" đồng thời 1 đơn vị dư và 1 đơn vị thiếu.

Từng đoạn:
1. `arr = [2*i+1 ...]` — dựng mảng.
2. `t = sum(arr) // n` — trung bình (luôn nguyên, bằng `n`).
3. `sum(abs(x - t) ...) // 2` — tổng lệch chia đôi.

In [13]:
def min_ops_mean(n):
    arr = [2 * i + 1 for i in range(n)]
    t = sum(arr) // n
    return sum(abs(x - t) for x in arr) // 2

print([min_ops_mean(n) for n, _ in examples], "mong đợi", [w for _, w in examples])

[2, 9] mong đợi [2, 9]


Code nộp LeetCode cho cách 2:

In [14]:
class SolutionMean:
    def minOperations(self, n: int) -> int:
        arr = [2 * i + 1 for i in range(n)]
        t = sum(arr) // n
        return sum(abs(x - t) for x in arr) // 2

print(SolutionMean().minOperations(3), SolutionMean().minOperations(6))

2 9


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho mảng `arr` (có thể bỏ mảng nếu sinh giá trị trực tiếp).

### 7.2 Cách 3 — Chỉ nửa đầu, không cần mảng, `O(n)` time `O(1)` space

Vì đích là `n` và nửa đầu toàn số `< n` (hoặc `= n` ở phần tử giữa khi `n` lẻ — nhưng phần tử giữa không thuộc `range(n//2)`), ta cộng `n − (2i+1)` cho `i` từ `0` đến `n//2 − 1`.

In [15]:
def min_ops_half(n):
    total = 0
    for i in range(n // 2):
        total += n - (2 * i + 1)
    return total

print([min_ops_half(n) for n in range(1, 11)])

[0, 1, 2, 4, 6, 9, 12, 16, 20, 25]


Code nộp LeetCode cho cách 3:

In [16]:
class SolutionHalf:
    def minOperations(self, n: int) -> int:
        return sum(n - (2 * i + 1) for i in range(n // 2))

print(SolutionHalf().minOperations(3), SolutionHalf().minOperations(6))

2 9


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.3 Cách 4 — Công thức đóng `n * n // 4`, `O(1)`

Từ mục 6: tổng phần thiếu là tổng các số cách nhau 2:
- `n` chẵn: `1 + 3 + ... + (n−1)` = tổng `n/2` số lẻ đầu = `(n/2)² = n²/4`.
- `n` lẻ: `2 + 4 + ... + (n−1)` = `2 · (1 + 2 + ... + (n−1)/2)` = `((n−1)/2)·((n+1)/2)` = `(n² − 1)/4`.

Hợp nhất: `⌊n²/4⌋` = `n * n // 4`.

In [17]:
for n in range(1, 12):
    if n % 2 == 0:
        f = (n // 2) ** 2
    else:
        f = ((n - 1) // 2) * ((n + 1) // 2)
    print(f"n={n:>2}: công thức tách chẵn/lẻ={f:>3}, n*n//4={n*n//4:>3}", "OK" if f == n * n // 4 else "LỆCH")

n= 1: công thức tách chẵn/lẻ=  0, n*n//4=  0 OK
n= 2: công thức tách chẵn/lẻ=  1, n*n//4=  1 OK
n= 3: công thức tách chẵn/lẻ=  2, n*n//4=  2 OK
n= 4: công thức tách chẵn/lẻ=  4, n*n//4=  4 OK
n= 5: công thức tách chẵn/lẻ=  6, n*n//4=  6 OK
n= 6: công thức tách chẵn/lẻ=  9, n*n//4=  9 OK
n= 7: công thức tách chẵn/lẻ= 12, n*n//4= 12 OK
n= 8: công thức tách chẵn/lẻ= 16, n*n//4= 16 OK
n= 9: công thức tách chẵn/lẻ= 20, n*n//4= 20 OK
n=10: công thức tách chẵn/lẻ= 25, n*n//4= 25 OK
n=11: công thức tách chẵn/lẻ= 30, n*n//4= 30 OK


Code nộp LeetCode cho cách 4:

In [18]:
class SolutionFormula:
    def minOperations(self, n: int) -> int:
        return n * n // 4

print(SolutionFormula().minOperations(3), SolutionFormula().minOperations(6), SolutionFormula().minOperations(10**4))

2 9 25000000


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

Một dạng tương đương hay gặp: `(n // 2) * ((n + 1) // 2)` — tích "số phần tử nửa dưới" và "số phần tử nửa trên làm tròn lên".

In [19]:
print(all((n // 2) * ((n + 1) // 2) == n * n // 4 for n in range(1, 10**4 + 1)))

True


## 8. TRACE từng bước

### 8.1 Trace cách 3 (nửa đầu) với `n = 6`

| i | arr[i] = 2i+1 | n − arr[i] | total |
|---|---------------|-----------|-------|
| 0 | 1 | 5 | 5 |
| 1 | 3 | 3 | 8 |
| 2 | 5 | 1 | 9 |

Dừng vì `n // 2 = 3`. Đáp án `9`.

In [20]:
def trace_half(n):
    total = 0
    print(f"n = {n}, đích = {n}, duyệt i = 0..{n//2 - 1}")
    for i in range(n // 2):
        need = n - (2 * i + 1)
        total += need
        print(f"  i={i}: arr[i]={2*i+1:>2}, cần thêm {need:>2}, total={total}")
    print("  => đáp án", total, "| n*n//4 =", n * n // 4)
    return total

trace_half(6)
trace_half(7)

n = 6, đích = 6, duyệt i = 0..2
  i=0: arr[i]= 1, cần thêm  5, total=5
  i=1: arr[i]= 3, cần thêm  3, total=8
  i=2: arr[i]= 5, cần thêm  1, total=9
  => đáp án 9 | n*n//4 = 9
n = 7, đích = 7, duyệt i = 0..2
  i=0: arr[i]= 1, cần thêm  6, total=6
  i=1: arr[i]= 3, cần thêm  4, total=10
  i=2: arr[i]= 5, cần thêm  2, total=12
  => đáp án 12 | n*n//4 = 12


12

### 8.2 Trace mô phỏng ghép cặp với `n = 4`

In trạng thái mảng sau mỗi thao tác để thấy mỗi bước chuyển đúng 1 đơn vị từ phần tử bên phải sang bên trái của cặp.

In [21]:
def trace_simulation(n):
    arr = [2 * i + 1 for i in range(n)]
    print("ban đầu:", arr)
    ops = 0
    i, j = 0, n - 1
    while i < j:
        while arr[i] < arr[j]:
            arr[j] -= 1
            arr[i] += 1
            ops += 1
            print(f"  thao tác {ops}: x={j} -> y={i}: {arr}")
        i += 1
        j -= 1
    print("tổng thao tác:", ops)
    return ops

trace_simulation(4)

ban đầu: [1, 3, 5, 7]
  thao tác 1: x=3 -> y=0: [2, 3, 5, 6]
  thao tác 2: x=3 -> y=0: [3, 3, 5, 5]
  thao tác 3: x=3 -> y=0: [4, 3, 5, 4]
  thao tác 4: x=2 -> y=1: [4, 4, 4, 4]
tổng thao tác: 4


4

In [22]:
trace_simulation(5)

ban đầu: [1, 3, 5, 7, 9]
  thao tác 1: x=4 -> y=0: [2, 3, 5, 7, 8]
  thao tác 2: x=4 -> y=0: [3, 3, 5, 7, 7]
  thao tác 3: x=4 -> y=0: [4, 3, 5, 7, 6]
  thao tác 4: x=4 -> y=0: [5, 3, 5, 7, 5]
  thao tác 5: x=3 -> y=1: [5, 4, 5, 6, 5]
  thao tác 6: x=3 -> y=1: [5, 5, 5, 5, 5]
tổng thao tác: 6


6

## 9. Code nộp LeetCode

Bản nộp ngắn nhất và nhanh nhất là công thức đóng. Chữ ký giữ nguyên `minOperations(self, n: int) -> int`.

In [23]:
class Solution:
    def minOperations(self, n: int) -> int:
        # Đích bắt buộc là trung bình = n; đáp án = tổng phần thiếu của nửa đầu = floor(n^2 / 4)
        return n * n // 4

for n, want in examples:
    got = Solution().minOperations(n)
    print(n, got, want)
    assert got == want

3 2 2
6 9 9


## 10. Vì sao thuật toán đúng

1. **Đích là duy nhất.** Thao tác giữ nguyên tổng. Trạng thái cuối có mọi phần tử bằng `t` ⇒ `n·t = n²` ⇒ `t = n`.
2. **Cận dưới.** Đặt `E(arr) = Σ max(0, arr[i] − n)`. Một thao tác chỉ trừ 1 ở **một** vị trí, nên `E` giảm tối đa 1 (có thể không giảm, hoặc thậm chí tăng nếu cộng vào phần tử đã ≥ n). Trạng thái cuối có `E = 0`. Do đó số thao tác ≥ `E(arr ban đầu)`.
3. **Cận trên.** Khi chưa xong, tồn tại phần tử `> n` và phần tử `< n` (vì tổng = `n²`). Chuyển 1 đơn vị từ cái thứ nhất sang cái thứ hai làm `E` giảm đúng 1. Lặp lại ⇒ đúng `E` thao tác.
4. **Tính `E`.** Phần tử `> n` là nửa sau; theo đối xứng `arr[i] + arr[n−1−i] = 2n`, phần dư nửa sau = phần thiếu nửa đầu = `Σ_{i<n//2}(n − 2i − 1) = ⌊n²/4⌋` (mục 7.3).

Vậy `n * n // 4` vừa là cận dưới vừa đạt được ⇒ là đáp án tối thiểu.

In [24]:
# Kiểm tra số học: E (định nghĩa) == n*n//4 cho mọi n trong ràng buộc
ok = all(sum(max(0, 2 * i + 1 - n) for i in range(n)) == n * n // 4 for n in range(1, 2001))
print("E == n*n//4 với n = 1..2000:", ok)

E == n*n//4 với n = 1..2000: True


## 11. Phân tích Time / Space

Gọi `n` là độ dài mảng (tham số đầu vào).

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| Mô phỏng max/min | `O(n³)` | `O(n)` | `n²/4` thao tác × `O(n)` tìm max/min |
| Mô phỏng ghép cặp | `O(n²)` | `O(n)` | mỗi thao tác `O(1)` |
| Tổng lệch / 2 | `O(n)` | `O(n)` | dựng mảng |
| Nửa đầu | `O(n)` | `O(1)` | không dựng mảng |
| Công thức | `O(1)` | `O(1)` | bản nộp |

Không có khác biệt average/worst: mọi cách đều tất định, chỉ phụ thuộc `n`. Với `n ≤ 10⁴`, đáp án tối đa `25 000 000` — vừa kiểu `int` 32 bit.

In [25]:
for name, f in [("mean", min_ops_mean), ("half", min_ops_half), ("formula", lambda n: n * n // 4)]:
    t0 = time.perf_counter()
    for n in range(1, 2001):
        f(n)
    print(f"{name:8s}: n = 1..2000 trong {time.perf_counter() - t0:.3f}s")

mean    : n = 1..2000 trong 0.211s
half    : n = 1..2000 trong 0.062s
formula : n = 1..2000 trong 0.000s


## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1`, `n = 2`, `n` lớn nhất), đối chiếu mọi cách với mô phỏng trên nhiều `n`, và kiểm tra ngẫu nhiên có seed.

In [26]:
METHODS = {
    "simulate_pairs": simulate_pairs,
    "mean": min_ops_mean,
    "half": min_ops_half,
    "SolutionMean": lambda n: SolutionMean().minOperations(n),
    "SolutionHalf": lambda n: SolutionHalf().minOperations(n),
    "SolutionFormula": lambda n: SolutionFormula().minOperations(n),
    "Solution": lambda n: Solution().minOperations(n),
}
# ví dụ đề + giá trị nhỏ đã biết
for n, want in list(examples) + list(small_known.items()):
    for name, f in METHODS.items():
        assert f(n) == want, (name, n, f(n), want)
print("ví dụ đề + giá trị nhỏ: OK")

ví dụ đề + giá trị nhỏ: OK


In [27]:
# edge case
assert Solution().minOperations(1) == 0      # mảng 1 phần tử đã bằng nhau
assert Solution().minOperations(2) == 1      # [1,3] -> [2,2]
assert Solution().minOperations(10**4) == 25_000_000
assert isinstance(Solution().minOperations(7), int)
print("edge case: OK")

edge case: OK


In [28]:
# đối chiếu với mô phỏng naive (chậm) cho n nhỏ, mô phỏng ghép cặp cho n vừa
for n in range(1, 31):
    ref = simulate_naive(n)
    for name, f in METHODS.items():
        assert f(n) == ref, (name, n)
for n in range(31, 301):
    ref = simulate_pairs(n)
    assert min_ops_half(n) == min_ops_mean(n) == Solution().minOperations(n) == ref
print("n = 1..300: mọi cách khớp mô phỏng")

n = 1..300: mọi cách khớp mô phỏng


In [29]:
# ngẫu nhiên có seed, trong ràng buộc 1 <= n <= 10^4
rng = random.Random(1551)
for _ in range(300):
    n = rng.randint(1, 10**4)
    assert min_ops_half(n) == min_ops_mean(n) == Solution().minOperations(n), n
print("300 giá trị n ngẫu nhiên (seed 1551): OK")

300 giá trị n ngẫu nhiên (seed 1551): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng chia thực `/` thay vì `//`

`n * n / 4` trả `float`, sai kiểu và sai giá trị khi `n` lẻ (`9/4 = 2.25`).

In [30]:
def wrong_float(n):
    return n * n / 4          # SAI: float

for n in [3, 5, 6]:
    print(f"n={n}: sai = {wrong_float(n)!r}, đúng = {n * n // 4}")

n=3: sai = 2.25, đúng = 2
n=5: sai = 6.25, đúng = 6
n=6: sai = 9.0, đúng = 9


Sửa: `return n * n // 4`.

### Lỗi 2 — Cộng tổng khoảng cách nhưng quên chia 2

`Σ|arr[i] − n|` đếm cả phần dư lẫn phần thiếu, mà một thao tác sửa cả hai cùng lúc ⇒ kết quả gấp đôi.

In [31]:
def wrong_no_half(n):
    return sum(abs(2 * i + 1 - n) for i in range(n))   # SAI: thiếu // 2

for n, want in examples:
    print(f"n={n}: sai = {wrong_no_half(n)}, đúng = {want}")

n=3: sai = 4, đúng = 2
n=6: sai = 18, đúng = 9


Sửa: `sum(abs(2*i + 1 - n) for i in range(n)) // 2`.

### Lỗi 3 — Chọn đích là "phần tử giữa" `arr[n // 2]`

Khi `n` lẻ thì `arr[n//2] = n` (đúng). Nhưng khi `n` chẵn thì `arr[n//2] = n + 1` ≠ trung bình ⇒ đích sai, kết quả sai (và thậm chí không thể đạt được vì tổng không khớp).

In [32]:
def wrong_median_target(n):
    arr = [2 * i + 1 for i in range(n)]
    t = arr[n // 2]                                   # SAI khi n chẵn
    return sum(max(0, x - t) for x in arr)

for n in [3, 4, 6, 8]:
    print(f"n={n}: đích sai={[2*i+1 for i in range(n)][n//2]}, kết quả sai={wrong_median_target(n)}, đúng={n*n//4}")

n=3: đích sai=3, kết quả sai=2, đúng=2
n=4: đích sai=5, kết quả sai=2, đúng=4
n=6: đích sai=7, kết quả sai=6, đúng=9
n=8: đích sai=9, kết quả sai=12, đúng=16


Sửa: đích luôn là `sum(arr) // n = n`.

### Lỗi 4 — Duyệt nửa đầu nhưng viết ngược dấu

Viết `(2*i + 1) − n` thay vì `n − (2*i + 1)` ⇒ cộng toàn số âm.

In [33]:
def wrong_sign(n):
    return sum((2 * i + 1) - n for i in range(n // 2))   # SAI dấu

for n, want in examples:
    print(f"n={n}: sai = {wrong_sign(n)}, đúng = {want}")

n=3: sai = -2, đúng = 2
n=6: sai = -9, đúng = 9


Sửa: `n - (2*i + 1)` (đích trừ giá trị nhỏ hơn), hoặc dùng `abs`.

### Lỗi 5 — Duyệt `range(n // 2 + 1)` khi `n` chẵn

Lấy dư một phần tử của nửa sau (giá trị `n + 1`) ⇒ cộng thêm `−1`.

In [34]:
def wrong_range(n):
    return sum(n - (2 * i + 1) for i in range(n // 2 + 1))   # SAI: thừa 1 phần tử

for n in [4, 5, 6]:
    print(f"n={n}: sai = {wrong_range(n)}, đúng = {n*n//4}")

n=4: sai = 3, đúng = 4
n=5: sai = 6, đúng = 6
n=6: sai = 8, đúng = 9


Sửa: `range(n // 2)`. (Khi `n` lẻ tình cờ đúng vì phần tử giữa có độ lệch 0 — kiểu lỗi "ăn may" rất nguy hiểm.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chuyển 1 đơn vị giữa hai phần tử"** ⇒ tổng bất biến ⇒ đích = trung bình ⇒ đáp án = tổng phần dư (hoặc `Σ|lệch|/2`).
- Nếu thao tác là **"+1 / −1 một phần tử"** (không bảo toàn tổng) thì đích tối ưu là **trung vị**, không phải trung bình (bài 462).
- Mảng là **cấp số cộng** ⇒ đối xứng quanh trung bình ⇒ chỉ cần nửa đầu; tổng của nửa đầu thường có công thức đóng.
- Gặp bài chỉ có một số `n` làm input: tính tay `n = 1..6`, tra dãy số (`0, 1, 2, 4, 6, 9, 12, 16...` = `⌊n²/4⌋`, "quarter-squares").
- Luôn kiểm tra `n` chẵn và lẻ riêng khi rút công thức.

In [35]:
print("Dãy ⌊n²/4⌋:", [n * n // 4 for n in range(1, 13)])
print("Hiệu liên tiếp:", [((n + 1) ** 2 // 4) - (n * n // 4) for n in range(1, 12)], "= ⌊(n+1)/2⌋")

Dãy ⌊n²/4⌋: [0, 1, 2, 4, 6, 9, 12, 16, 20, 25, 30, 36]
Hiệu liên tiếp: [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6] = ⌊(n+1)/2⌋


## 15. Biến thể của bài

1. **Mảng bất kỳ** (không chỉ số lẻ), cùng thao tác: đáp án `Σ max(0, a[i] − avg)` nếu `sum % n == 0`, ngược lại `−1`.
2. **Cấp số cộng tổng quát** `arr[i] = c + d·i`: trung bình `c + d(n−1)/2`, cần là số nguyên.
3. **Thao tác ±1 một phần tử** (Minimum Moves to Equal Array Elements II, LC 462): đích là trung vị, đáp án `Σ|a[i] − median|`.
4. **Chỉ được chuyển giữa hai phần tử kề nhau** (kiểu "Super Washing Machines", LC 517): đáp án liên quan tới max của tiền tố phần dư.
5. **Mỗi thao tác chuyển k đơn vị**: đáp án = `ceil` theo từng phần tử, cần xét chia hết.

In [36]:
def ops_general(a):
    s, n = sum(a), len(a)
    if s % n:
        return -1
    t = s // n
    return sum(max(0, x - t) for x in a)

def moves_median(a):            # biến thể 3 (LC 462)
    m = sorted(a)[len(a) // 2]
    return sum(abs(x - m) for x in a)

print(ops_general([1, 3, 5]), ops_general([4, 0, 2]), ops_general([1, 2]))
print(moves_median([1, 2, 3]), moves_median([1, 10, 2, 9]))

2 2 -1
2 16


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Mô phỏng | chuyển từng đơn vị lớn nhất → nhỏ nhất | `O(n²)`–`O(n³)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. Tổng lệch / 2 | đích = trung bình | `O(n)` | `O(n)` | mảng bất kỳ (biến thể 1) |
| 3. Nửa đầu | đối xứng, chỉ cộng phần thiếu | `O(n)` | `O(1)` | khi chưa nhớ công thức |
| 4. Công thức `n*n//4` | tổng cấp số cộng | `O(1)` | `O(1)` | **bản nộp** |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minOperations(4)` và `Solution().minOperations(9)` trả về gì?

**Bài 2 (tính tay):** Với `n = 7`, liệt kê `arr`, phần thiếu của từng phần tử nửa đầu và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_ops_general(a)`: số thao tác tối thiểu (cùng loại thao tác của đề) để mọi phần tử của mảng `a` bất kỳ bằng nhau; trả `-1` nếu không thể.

**Bài 4 (tự viết lại):** Viết `ex4_two_pointers(n)` tính đáp án bằng hai con trỏ `i = 0`, `j = n−1`, cộng `(arr[j] − arr[i]) // 2` cho mỗi cặp — không dùng công thức.

**Bài 5 (mini challenge):** Viết `ex5_arith(n, c, d)` cho mảng `arr[i] = c + d·i` (`d ≥ 0`): trả số thao tác tối thiểu, hoặc `-1` nếu không thể làm bằng nhau. Yêu cầu `O(1)`.

In [37]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_ops_general(a):
    s, n = sum(a), len(a)
    # TODO: nếu s không chia hết cho n thì trả -1
    # TODO: t = trung bình; trả tổng phần dư max(0, x - t)
    pass

In [38]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_two_pointers(n):
    # TODO
    raise NotImplementedError

def ex5_arith(n, c, d):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dùng `n*n//4`. (2) `16 // 4` và `81 // 4`. (3) `4` và `20`.

**Bài 2.** (1) `arr = [1,3,5,7,9,11,13]`, đích `7`. (2) Nửa đầu: `1, 3, 5` thiếu `6, 4, 2`. (3) Đáp án `12 = 49 // 4`.

**Bài 3.** (1) Tổng bất biến ⇒ đích phải là `s / n`. (2) `if s % n: return -1`. (3) `return sum(max(0, x - s // n) for x in a)`.

**Bài 4.** (1) Cặp `(i, n−1−i)` có tổng `2n`. (2) Mỗi cặp cần `(arr[j] − arr[i]) // 2` thao tác. (3) Dừng khi `i >= j`.

**Bài 5.** (1) Trung bình là `c + d(n−1)/2` ⇒ cần `d·(n−1)` chẵn. (2) Phần thiếu nửa đầu: `d·((n−1)/2 − i)` với `i < n//2`. (3) Tổng `= d · Σ_{i<n//2} ((n−1) − 2i) / 2 = d · (n*n//4) / 2` — kiểm chứng bằng `ops_general` trên mảng dựng sẵn.

## 19. Đáp án tham khảo cho bài tập

In [39]:
# Bài 1
a1 = (Solution().minOperations(4), Solution().minOperations(9))
print("Bài 1:", a1)
assert a1 == (4, 20)

# Bài 2
arr7 = [2 * i + 1 for i in range(7)]
need7 = [7 - x for x in arr7[: 7 // 2]]
print("Bài 2:", arr7, "thiếu:", need7, "đáp án:", sum(need7))
assert sum(need7) == 12 == Solution().minOperations(7)

Bài 1: (4, 20)
Bài 2: [1, 3, 5, 7, 9, 11, 13] thiếu: [6, 4, 2] đáp án: 12


In [40]:
# Bài 3
def ex3_ops_general_sol(a):
    s, n = sum(a), len(a)
    if s % n:
        return -1
    t = s // n
    return sum(max(0, x - t) for x in a)

def brute_general(a):
    # mô phỏng: chuyển từ max sang min cho tới khi bằng nhau
    if sum(a) % len(a):
        return -1
    a = list(a)
    ops = 0
    while max(a) != min(a):
        a[a.index(max(a))] -= 1
        a[a.index(min(a))] += 1
        ops += 1
    return ops

rng3 = random.Random(3)
for _ in range(500):
    a = [rng3.randint(0, 12) for _ in range(rng3.randint(1, 7))]
    assert ex3_ops_general_sol(a) == brute_general(a), a
assert all(ex3_ops_general_sol([2 * i + 1 for i in range(n)]) == n * n // 4 for n in range(1, 200))
print("Bài 3: OK")

Bài 3: OK


In [41]:
# Bài 4
def ex4_two_pointers_sol(n):
    i, j, total = 0, n - 1, 0
    while i < j:
        total += ((2 * j + 1) - (2 * i + 1)) // 2
        i += 1
        j -= 1
    return total

assert all(ex4_two_pointers_sol(n) == n * n // 4 for n in range(1, 3001))
print("Bài 4: OK", [ex4_two_pointers_sol(n) for n in range(1, 9)])

Bài 4: OK [0, 1, 2, 4, 6, 9, 12, 16]


In [42]:
# Bài 5
def ex5_arith_sol(n, c, d):
    if d * (n - 1) % 2:
        return -1
    # tổng phần thiếu = d * Σ_{i<n//2} ((n-1)/2 - i) = d * (n*n//4) / 2
    return d * (n * n // 4) // 2

rng5 = random.Random(5)
for _ in range(1500):
    n = rng5.randint(1, 40)
    c = rng5.randint(0, 20)
    d = rng5.randint(0, 9)
    a = [c + d * i for i in range(n)]
    assert ex5_arith_sol(n, c, d) == ex3_ops_general_sol(a), (n, c, d)
assert ex5_arith_sol(6, 1, 2) == 9 and ex5_arith_sol(3, 1, 2) == 2
print("Bài 5: OK, ví dụ:", ex5_arith_sol(5, 0, 3), ex5_arith_sol(4, 0, 3))

Bài 5: OK, ví dụ: 9 -1


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def minOperations(self, n: int) -> int:
        return n * n // 4            # = tổng phần thiếu của nửa đầu so với trung bình n
```

**Template "chuyển đơn vị giữa các phần tử cho bằng nhau":**

```python
def min_transfer_ops(a):
    s, n = sum(a), len(a)
    if s % n:            # tổng bất biến -> phải chia hết
        return -1
    t = s // n           # đích = trung bình
    return sum(max(0, x - t) for x in a)   # = sum(|x - t|) // 2
```

**Checklist trước khi submit:**
- [ ] Đích là **trung bình** (tổng bất biến), không phải trung vị hay phần tử giữa.
- [ ] Dùng `//` để trả `int`.
- [ ] Nếu cộng `|lệch|` của toàn mảng thì **chia 2**.
- [ ] Vòng nửa đầu là `range(n // 2)`, dấu `n − arr[i]`.
- [ ] Thử nhanh: `n = 1 → 0`, `n = 2 → 1`, `n = 3 → 2`, `n = 6 → 9`.